In [2]:
# ============== DATASET CELL ==============

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Paths
train_folder = r"C:\fish_classification\Fish_Dataset\train_augmented"
val_folder   = r"C:\fish_classification\Fish_Dataset\val"

# Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Dataset
train_data = datasets.ImageFolder(train_folder, transform=train_transforms)
val_data   = datasets.ImageFolder(val_folder, transform=val_transforms)

# DataLoaders
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False)

print("Train images:", len(train_data))
print("Val images:", len(val_data))
print("Classes:", train_data.classes)


Train images: 43740
Val images: 2751
Classes: ['Bangus', 'Big Head Carp', 'Black Spotted Barb', 'Catfish', 'Climbing Perch', 'Fourfinger Threadfin', 'Freshwater Eel', 'Glass Perchlet', 'Goby', 'Gold Fish', 'Gourami', 'Grass Carp', 'Green Spotted Puffer', 'Indian Carp', 'Indo-Pacific Tarpon', 'Jaguar Gapote', 'Janitor Fish', 'Knifefish', 'Long-Snouted Pipefish', 'Mosquito Fish', 'Mudfish', 'Mullet', 'Pangasius', 'Perch', 'Scat Fish', 'Silver Barb', 'Silver Carp', 'Silver Perch', 'Snakehead', 'Tenpounder', 'Tilapia']


In [ ]:
# ================= FAST TRAINING MODE =================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

print("FAST MODE TRAINING STARTED")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Model (NEW API, no warning)
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
num_classes = len(train_data.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    correct = 0
    total = 0

    print(f"\nEpoch {epoch+1}/{num_epochs} started")

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_acc = 100 * correct / total

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(f"Epoch {epoch+1} DONE | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

# Save model
save_path = r"C:\fish_classification\fish_model_fast.pth"
torch.save(model.state_dict(), save_path)

print("\nFAST MODE TRAINING COMPLETE")
print("Model saved at:", save_path)


FAST MODE TRAINING STARTED
Using device: cpu

Epoch 1/3 started
Epoch 1 | Batch 0/1367 | Loss: 3.6312
Epoch 1 | Batch 50/1367 | Loss: 2.7108
Epoch 1 | Batch 100/1367 | Loss: 2.0464
Epoch 1 | Batch 150/1367 | Loss: 1.6370
Epoch 1 | Batch 200/1367 | Loss: 1.8979
Epoch 1 | Batch 250/1367 | Loss: 1.6362
Epoch 1 | Batch 300/1367 | Loss: 1.3867
Epoch 1 | Batch 350/1367 | Loss: 1.7219


In [ ]:
torch.cuda.is_available()
